In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.business_transformation.loans')
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. READ BUSINESS TRANSFORMATION DATA
# ============================================================

source_df = spark.read.table(
    "fintech360.business_transformation.loans"
)

# ============================================================
# 2. DEDUPLICATE LOANS
# Append-mode upstream processing may create duplicate loan IDs.
# Retain the latest updated record, then latest transformation.
# ============================================================

window_spec = (
    Window
    .partitionBy("loan_id")
    .orderBy(
        F.col("updated_at").desc_nulls_last(),
        F.col("business_transformed_at").desc_nulls_last()
    )
)

df = (
    source_df
    .withColumn("_gold_row_number", F.row_number().over(window_spec))
    .filter(F.col("_gold_row_number") == 1)
    .drop("_gold_row_number")
)

# ============================================================
# 3. STANDARDIZE GOLD BUSINESS FIELDS
# ============================================================

df = (df
    .withColumn("gold_loan_status", F.upper(F.trim(F.col("loan_status"))))
    .withColumn("gold_loan_source", F.upper(F.trim(F.col("loan_source"))))
    .withColumn("gold_loan_purpose", F.upper(F.trim(F.col("loan_purpose"))))
    .withColumn("gold_application_channel", F.upper(F.trim(F.col("application_channel"))))
    .withColumn("gold_principal_amount", F.col("principal_amount").cast("double"))
    .withColumn("gold_interest_rate_pct", F.col("annual_interest_rate_pct").cast("double"))
    .withColumn("gold_tenure_months", F.col("tenure_months").cast("int"))
)

# ============================================================
# 4. GOLD LOAN LIFECYCLE FLAGS
# Derive flags from current loan status and disbursement timestamp.
# ============================================================

df = (df
    .withColumn(
        "gold_application_flag",
        F.when(
            F.col("gold_loan_status").isin(
                "APPLIED", "APPROVED", "DISBURSED", "REJECTED", "CLOSED"
            ),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_approved_flag",
        F.when(
            F.col("gold_loan_status").isin(
                "APPROVED", "DISBURSED", "CLOSED"
            ),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_disbursed_flag",
        F.when(
            F.col("disbursement_ts").isNotNull() &
            F.col("gold_loan_status").isin("DISBURSED", "CLOSED"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_rejected_flag",
        F.when(F.col("gold_loan_status") == "REJECTED", 1)
         .otherwise(0)
    )
    .withColumn(
        "gold_closed_flag",
        F.when(F.col("gold_loan_status") == "CLOSED", 1)
         .otherwise(0)
    )
    .withColumn(
        "gold_approved_pending_disbursal_flag",
        F.when(
            (F.col("gold_loan_status") == "APPROVED") &
            F.col("disbursement_ts").isNull(),
            1
        ).otherwise(0)
    )
)

# ============================================================
# 5. GOLD FINANCIAL MEASURES
# ============================================================

df = (df
    .withColumn(
        "gold_disbursed_principal",
        F.when(
            F.col("gold_disbursed_flag") == 1,
            F.col("gold_principal_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "gold_approved_pipeline_amount",
        F.when(
            F.col("gold_approved_pending_disbursal_flag") == 1,
            F.col("gold_principal_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "gold_rejected_application_amount",
        F.when(
            F.col("gold_rejected_flag") == 1,
            F.col("gold_principal_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "gold_estimated_interest",
        F.when(
            F.col("gold_disbursed_flag") == 1,
            F.col("estimated_simple_interest_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "gold_estimated_principal_plus_interest",
        F.when(
            F.col("gold_disbursed_flag") == 1,
            F.col("estimated_principal_plus_simple_interest")
        ).otherwise(F.lit(0.0))
    )
)

# ============================================================
# 6. PORTFOLIO SEGMENTATION
# ============================================================

df = (df
    .withColumn(
        "gold_portfolio_segment",
        F.when(
            F.col("gold_disbursed_flag") == 1,
            "DISBURSED_PORTFOLIO"
        ).when(
            F.col("gold_approved_pending_disbursal_flag") == 1,
            "APPROVED_PIPELINE"
        ).when(
            F.col("gold_rejected_flag") == 1,
            "REJECTED_APPLICATIONS"
        ).when(
            F.col("gold_loan_status") == "APPLIED",
            "APPLICATION_PIPELINE"
        ).otherwise("OTHER_OR_UNCLASSIFIED")
    )
    .withColumn(
        "gold_principal_band",
        F.when(F.col("gold_principal_amount") < 5000, "BELOW_5000")
         .when(F.col("gold_principal_amount") < 10000, "5000_TO_9999")
         .when(F.col("gold_principal_amount") < 20000, "10000_TO_19999")
         .when(F.col("gold_principal_amount") < 50000, "20000_TO_49999")
         .otherwise("50000_PLUS")
    )
    .withColumn(
        "gold_tenure_segment",
        F.when(F.col("gold_tenure_months") <= 3, "UP_TO_3_MONTHS")
         .when(F.col("gold_tenure_months") <= 6, "4_TO_6_MONTHS")
         .when(F.col("gold_tenure_months") <= 12, "7_TO_12_MONTHS")
         .otherwise("OVER_12_MONTHS")
    )
)

# ============================================================
# 7. FUNNEL AND TURNAROUND METRICS
# ============================================================

df = (df
    .withColumn(
        "gold_disbursal_tat_days",
        F.col("application_to_disbursal_days")
    )
    .withColumn(
        "gold_disbursal_tat_hours",
        F.col("application_to_disbursal_hours")
    )
    .withColumn(
        "gold_fast_disbursal_flag",
        F.when(
            (F.col("gold_disbursed_flag") == 1) &
            (F.col("application_to_disbursal_hours") <= 24) &
            (F.col("application_to_disbursal_hours") >= 0),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_funnel_stage",
        F.when(F.col("gold_rejected_flag") == 1, "REJECTED")
         .when(F.col("gold_disbursed_flag") == 1, "DISBURSED")
         .when(F.col("gold_approved_flag") == 1, "APPROVED")
         .when(F.col("gold_loan_status") == "APPLIED", "APPLIED")
         .otherwise("OTHER")
    )
)

# ============================================================
# 8. BUSINESS REVIEW SIGNALS
# ============================================================

df = df.withColumn(
    "gold_review_category",
    F.when(
        F.col("maturity_before_disbursal_flag") == 1,
        "DATE_CONSISTENCY_REVIEW"
    ).when(
        F.col("gold_approved_pending_disbursal_flag") == 1,
        "APPROVED_NOT_DISBURSED"
    ).when(
        F.col("maturity_date_missing_flag") == 1,
        "MISSING_MATURITY_DATE"
    ).when(
        F.col("high_interest_rate_review_flag") == 1,
        "HIGH_INTEREST_RATE_REVIEW"
    ).otherwise("NO_BASIC_REVIEW_SIGNAL")
)

# ============================================================
# 9. GOLD DATA QUALITY CHECKS
# These checks are additional controls for the Gold layer.
# ============================================================

df = (df
    .withColumn(
        "gold_missing_customer_flag",
        F.when(F.col("customer_id").isNull(), 1).otherwise(0)
    )
    .withColumn(
        "gold_invalid_principal_flag",
        F.when(
            F.col("gold_principal_amount").isNull() |
            (F.col("gold_principal_amount") <= 0),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_missing_application_id_flag",
        F.when(
            F.col("application_id").isNull() |
            (F.trim(F.col("application_id")) == ""),
            1
        ).otherwise(0)
    )
)

# ============================================================
# 10. GOLD METADATA
# ============================================================

df = (df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_table", F.lit("fintech360.business_transformation.loans"))
)

# ============================================================
# 11. FINAL GOLD DATAFRAME
# ============================================================

gold_loans = df

display(gold_loans.limit(10))


In [0]:

# Create Gold schema
spark.sql("""
CREATE SCHEMA IF NOT EXISTS fintech360.gold
""")

# Write the deduplicated Gold loan dataset
(gold_loans.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.gold.loans"))

print("Gold loan transformation completed.")
print("Target: fintech360.gold.loans")
